# 06 Modeling — Late Delivery Risk Prediction

This notebook builds the machine learning workflow for SupplyGuard.

The objective is to predict whether an order is at risk of late delivery using only information available at or shortly after payment approval. This timing is important because the model is intended to support proactive operational decisions before the delivery outcome is known.

The workflow focuses on leakage-safe model evaluation, consistent preprocessing, class imbalance handling, and business-relevant metrics.

In [44]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.base import clone


pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [3]:
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_path = project_root / "data" / "processed" / "modeling_dataset.csv"

df = pd.read_csv(data_path)

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
display(df.head())

Rows: 96,470
Columns: 35


,order_id,is_late,purchase_month,purchase_day_of_week,purchase_hour,is_weekend_purchase,estimated_delivery_days,approval_delay_hours,customer_state,main_seller_state,same_state_order,cross_state_order,customer_seller_state_pair,customer_seller_distance_km,order_item_count,product_count,seller_count,total_item_price,total_freight_value,avg_item_price,max_item_price,total_order_item_value,freight_ratio,product_category_count,dominant_product_category,total_product_weight_g,max_product_weight_g,total_product_volume_cm3,max_product_volume_cm3,payment_count,payment_method_count,total_payment_value,avg_payment_value,max_payment_installments,main_payment_type
0,e481f51cbdc54678b7cc49136f2d6af7,0,10,0,10,0,16,0.178333,SP,SP,1,0,SP_SP,18.676576,1,1,1,29.99,8.72,29.99,29.99,38.71,0.225265,1,housewares,500.0,500.0,1976.0,1976.0,3.0,2.0,38.71,12.90,1.0,voucher
1,53cdb2fc8bc7dce0b6741e2150273451,0,7,1,20,0,20,30.713889,BA,SP,0,1,BA_SP,861.068703,1,1,1,118.70,22.76,118.70,118.70,141.46,0.160894,1,perfumery,400.0,400.0,4693.0,4693.0,1.0,1.0,141.46,141.46,1.0,boleto
2,47770eb9100c2d0c44946d9cf07ec65d,0,8,2,8,0,27,0.276111,GO,SP,0,1,GO_SP,514.535098,1,1,1,159.90,19.22,159.90,159.90,179.12,0.107302,1,auto,420.0,420.0,9576.0,9576.0,1.0,1.0,179.12,179.12,3.0,credit_card
3,949d5b44dbf5de918fe9c16f97b45f8a,0,11,5,19,1,27,0.298056,RN,MG,0,1,RN_MG,1821.871635,1,1,1,45.00,27.20,45.00,45.00,72.20,0.376731,1,pet_shop,450.0,450.0,6000.0,6000.0,1.0,1.0,72.20,72.20,1.0,credit_card
4,ad21c59c0840e6cb83a9ceb5573f8159,0,2,1,21,0,13,1.030556,SP,SP,1,0,SP_SP,29.623195,1,1,1,19.90,8.72,19.90,19.90,28.62,0.304682,1,stationery,250.0,250.0,11475.0,11475.0,1.0,1.0,28.62,28.62,1.0,credit_card


## Target Definition and Prediction Timing

The target variable is `is_late`, where:

- `0` = order delivered on time
- `1` = order delivered late

The official project definition uses date-only logic: an order is late only when the delivered date is later than the estimated delivery date.

The prediction moment for this notebook is shortly after payment approval. Therefore, payment-related variables, estimated delivery window, customer-seller geography and order/product attributes are valid features as long as they do not use the actual delivery outcome.

In [4]:
id_col = "order_id"
target_col = "is_late"

order_ids = df[id_col].copy()
y = df[target_col].copy()
X = df.drop(columns=[id_col, target_col])

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")

display(y.value_counts().rename_axis("is_late").reset_index(name="orders"))
display((y.value_counts(normalize=True) * 100).round(2).rename_axis("is_late").reset_index(name="order_share_pct"))

Feature matrix shape: (96470, 33)
Target shape: (96470,)


,is_late,orders
0,0,89936
1,1,6534


,is_late,order_share_pct
0,0,93.23
1,1,6.77


## Leakage Control

This notebook uses only features available at or shortly after payment approval.

The modeling dataset was created in the previous notebook as a leakage-safe, order-level dataset. No post-delivery information should be used as a predictive feature.

The following variables are excluded from the feature matrix:

- `order_id`: identifier kept only for traceability.
- `is_late`: target variable.

The following leakage-sensitive variables must not be used as model features:

- actual customer delivery date;
- delivery delay fields;
- review score and review comments;
- final delivery outcome variables;
- raw delivery timestamps;
- `order_status`;
- any information only known after delivery.

This restriction is necessary because the model is intended to predict late delivery risk before the delivery outcome is known.

In [9]:
leakage_keywords = ["delivered", "delivery_delay", "delay_days", "review", "order_status"]
potential_leakage_cols = [c for c in X.columns if any(k in c.lower() for k in leakage_keywords)]

display(pd.DataFrame({"potential_leakage_column": potential_leakage_cols}))

if potential_leakage_cols:
    raise ValueError("Potential leakage columns found. Review and remove them before modeling.")

,potential_leakage_column


## Feature Groups

Calendar-coded variables are treated as categorical features, while binary indicators are kept as 0/1.
`cross_state_order` is excluded because it is the exact complement of `same_state_order`.

In [13]:
excluded_feature_cols = ["cross_state_order"]
X_model = X.drop(columns=excluded_feature_cols)

calendar_categorical_features = ["purchase_month", "purchase_day_of_week", "purchase_hour"]
binary_features = ["is_weekend_purchase", "same_state_order"]

categorical_features = [
    "customer_state", "main_seller_state", "customer_seller_state_pair",
    "dominant_product_category", "main_payment_type",
    *calendar_categorical_features
]

numeric_features = [c for c in X_model.columns if c not in categorical_features + binary_features]

print(f"Model features: {X_model.shape[1]}")
print(f"Numeric features: {len(numeric_features)}")
print(f"Binary features: {len(binary_features)}")
print(f"Categorical features: {len(categorical_features)}")

Model features: 32
Numeric features: 22
Binary features: 2
Categorical features: 8


## Train/Validation/Test Split

The test set is kept untouched for final evaluation; model and threshold decisions are made on the validation set.

In [24]:
X_train_val, X_test, y_train_val, y_test, train_val_ids, test_ids = train_test_split(
    X_model, y, order_ids, test_size=0.2, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val, train_ids, val_ids = train_test_split(
    X_train_val, y_train_val, train_val_ids, test_size=0.25, random_state=42, stratify=y_train_val
)

split_summary = pd.DataFrame({
    "dataset": ["Full", "Train", "Validation", "Test"],
    "orders": [len(y), len(y_train), len(y_val), len(y_test)],
    "late_orders": [y.sum(), y_train.sum(), y_val.sum(), y_test.sum()],
    "late_rate_pct": [(y.mean() * 100).round(2), (y_train.mean() * 100).round(2), (y_val.mean() * 100).round(2), (y_test.mean() * 100).round(2)]
})

display(split_summary)

,dataset,orders,late_orders,late_rate_pct
0,Full,96470,6534,6.77
1,Train,57882,3920,6.77
2,Validation,19294,1307,6.77
3,Test,19294,1307,6.77


## Preprocessing

All imputing, scaling and encoding steps are fitted only through the modeling pipeline.

In [15]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])

binary_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
    ("bin", binary_transformer, binary_features)
])

## Baseline Model

A dummy classifier is used as the minimum benchmark for evaluating whether real models add predictive value.

In [25]:
def evaluate_classifier(model_name, model, X_eval, y_eval):
    y_pred = model.predict(X_eval)
    y_proba = model.predict_proba(X_eval)[:, 1]

    return {
        "model": model_name,
        "accuracy": accuracy_score(y_eval, y_pred),
        "precision": precision_score(y_eval, y_pred, zero_division=0),
        "recall": recall_score(y_eval, y_pred, zero_division=0),
        "f1_score": f1_score(y_eval, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_eval, y_proba),
        "pr_auc": average_precision_score(y_eval, y_proba)
    }

In [26]:
dummy_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", DummyClassifier(strategy="most_frequent"))
])

dummy_pipeline.fit(X_train, y_train)

baseline_results = pd.DataFrame([
    evaluate_classifier("Dummy Classifier", dummy_pipeline, X_val, y_val)
]).round(4)

display(baseline_results)

,model,accuracy,precision,recall,f1_score,roc_auc,pr_auc
0,Dummy Classifier,0.9323,0.0,0.0,0.0,0.5,0.0677


The dummy baseline confirms that accuracy is misleading in this imbalanced problem: predicting every order as on time reaches 93.23% accuracy but detects 0 late deliveries.

## Model Comparison

The first model comparison uses class-weighted models to account for the low late delivery rate.

In [27]:
models = {
    "Logistic Regression": LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(class_weight="balanced", n_estimators=200, random_state=42, n_jobs=-1),
    "Extra Trees": ExtraTreesClassifier(class_weight="balanced", n_estimators=200, random_state=42, n_jobs=-1)
}

fitted_pipelines = {}
model_results = []

for model_name, classifier in models.items():
    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("model", classifier)])
    pipeline.fit(X_train, y_train)
    fitted_pipelines[model_name] = pipeline
    model_results.append(evaluate_classifier(model_name, pipeline, X_val, y_val))

model_comparison_results = pd.concat([baseline_results, pd.DataFrame(model_results)], ignore_index=True).round(4)
display(model_comparison_results.sort_values("pr_auc", ascending=False))

,model,accuracy,precision,recall,f1_score,roc_auc,pr_auc
3,Random Forest,0.9328,0.8571,0.0092,0.0182,0.7563,0.2535
1,Logistic Regression,0.6950,0.1416,0.6917,0.2350,0.7628,0.2231
4,Extra Trees,0.9326,0.5385,0.0321,0.0606,0.7479,0.2214
2,Decision Tree,0.8812,0.1735,0.2005,0.1860,0.5655,0.0889
0,Dummy Classifier,0.9323,0.0000,0.0000,0.0000,0.5000,0.0677


Random Forest has the strongest PR-AUC, while Logistic Regression provides the strongest default-threshold recall. Both are kept for threshold analysis.

In [30]:
def threshold_metrics(model_name, pipeline, X_eval, y_eval, thresholds=np.arange(0.05, 0.51, 0.05)):
    y_proba = pipeline.predict_proba(X_eval)[:, 1]
    rows = []

    for threshold in thresholds:
        y_pred = (y_proba >= threshold).astype(int)
        rows.append({
            "model": model_name,
            "threshold": threshold,
            "precision": precision_score(y_eval, y_pred, zero_division=0),
            "recall": recall_score(y_eval, y_pred, zero_division=0),
            "f1_score": f1_score(y_eval, y_pred, zero_division=0),
            "predicted_late_orders": y_pred.sum(),
            "actual_late_orders": y_eval.sum()
        })

    return pd.DataFrame(rows)

In [31]:
threshold_results = pd.concat([
    threshold_metrics("Logistic Regression", fitted_pipelines["Logistic Regression"], X_val, y_val),
    threshold_metrics("Random Forest", fitted_pipelines["Random Forest"], X_val, y_val)
], ignore_index=True).round(4)

display(threshold_results.sort_values(["model", "threshold"]))

,model,threshold,precision,recall,f1_score,predicted_late_orders,actual_late_orders
0,Logistic Regression,0.05,0.0701,0.9962,0.1310,18572,1307
1,Logistic Regression,0.10,0.0733,0.9908,0.1364,17675,1307
2,Logistic Regression,0.15,0.0773,0.9778,0.1432,16538,1307
3,Logistic Regression,0.20,0.0827,0.9625,0.1524,15203,1307
4,Logistic Regression,0.25,0.0897,0.9388,0.1637,13682,1307
5,Logistic Regression,0.30,0.0972,0.8998,0.1754,12100,1307
6,Logistic Regression,0.35,0.1060,0.8585,0.1887,10585,1307
7,Logistic Regression,0.40,0.1154,0.8041,0.2018,9110,1307
8,Logistic Regression,0.45,0.1282,0.7536,0.2191,7685,1307
9,Logistic Regression,0.50,0.1416,0.6917,0.2350,6386,1307


Random Forest is selected for moderate tuning because it achieves the strongest validation PR-AUC. Its default threshold is too conservative, so threshold selection will be handled separately after tuning.

## Risk Ranking Check

Random Forest is also evaluated as a risk-ranking model to verify whether higher predicted risk scores concentrate more late deliveries.

In [32]:
rf_val_proba = fitted_pipelines["Random Forest"].predict_proba(X_val)[:, 1]

risk_lift = pd.DataFrame({"y_true": y_val.values, "risk_score": rf_val_proba})
risk_lift["risk_bucket"] = pd.qcut(risk_lift["risk_score"], q=10, labels=False, duplicates="drop") + 1

lift_summary = risk_lift.groupby("risk_bucket").agg(
    orders=("y_true", "size"),
    late_orders=("y_true", "sum"),
    late_rate=("y_true", "mean")
).reset_index().sort_values("risk_bucket", ascending=False)

lift_summary["late_rate_pct"] = (lift_summary["late_rate"] * 100).round(2)
lift_summary["late_capture_pct"] = (lift_summary["late_orders"] / risk_lift["y_true"].sum() * 100).round(2)

display(lift_summary)

,risk_bucket,orders,late_orders,late_rate,late_rate_pct,late_capture_pct
9,10,1792,470,0.262277,26.23,35.96
8,9,2008,262,0.130478,13.05,20.05
7,8,1672,114,0.068182,6.82,8.72
6,7,1833,106,0.057829,5.78,8.11
5,6,1571,67,0.042648,4.26,5.13
4,5,1947,71,0.036466,3.65,5.43
3,4,2259,77,0.034086,3.41,5.89
2,3,1231,35,0.028432,2.84,2.68
1,2,2437,62,0.025441,2.54,4.74
0,1,2544,43,0.016903,1.69,3.29


In [33]:
top_10_cutoff = np.quantile(rf_val_proba, 0.90)
top_10_pred = rf_val_proba >= top_10_cutoff

top_10_summary = pd.DataFrame([{
    "segment": "Top 10% highest-risk orders",
    "orders": top_10_pred.sum(),
    "late_orders": y_val[top_10_pred].sum(),
    "late_rate_pct": (y_val[top_10_pred].mean() * 100).round(2),
    "late_capture_pct": (y_val[top_10_pred].sum() / y_val.sum() * 100).round(2),
    "baseline_late_rate_pct": (y_val.mean() * 100).round(2)
}])

display(top_10_summary)

,segment,orders,late_orders,late_rate_pct,late_capture_pct,baseline_late_rate_pct
0,Top 10% highest-risk orders,1932,494,25.57,37.8,6.77


The model shows useful ranking behavior: the top 10% highest-risk validation orders contain a 25.57% late delivery rate, compared with the 6.77% baseline. This segment captures 37.8% of all late deliveries while reviewing only 10% of orders.

## Random Forest Tuning

In [36]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1))
])

rf_param_distributions = {
    "model__n_estimators": [200, 300, 400, 500],
    "model__max_depth": [8, 12, 16, 20, None],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 5, 10],
    "model__max_features": ["sqrt", "log2", 0.5],
    "model__bootstrap": [True, False]
}

rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_param_distributions,
    n_iter=20,
    scoring="average_precision",
    cv=3,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

rf_search.fit(X_train, y_train)

print(f"Best CV PR-AUC: {rf_search.best_score_:.4f}")
display(pd.DataFrame([rf_search.best_params_]))

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best CV PR-AUC: 0.2357


,model__n_estimators,model__min_samples_split,model__min_samples_leaf,model__max_features,model__max_depth,model__bootstrap
0,300,2,10,sqrt,16,False


In [37]:
tuned_rf_pipeline = rf_search.best_estimator_

tuned_rf_results = pd.DataFrame([
    evaluate_classifier("Random Forest Tuned", tuned_rf_pipeline, X_val, y_val)
]).round(4)

model_comparison_with_tuned = pd.concat([model_comparison_results, tuned_rf_results], ignore_index=True)
display(model_comparison_with_tuned.sort_values("pr_auc", ascending=False))

,model,accuracy,precision,recall,f1_score,roc_auc,pr_auc
3,Random Forest,0.9328,0.8571,0.0092,0.0182,0.7563,0.2535
5,Random Forest Tuned,0.7938,0.1775,0.5624,0.2698,0.7652,0.2384
1,Logistic Regression,0.6950,0.1416,0.6917,0.2350,0.7628,0.2231
4,Extra Trees,0.9326,0.5385,0.0321,0.0606,0.7479,0.2214
2,Decision Tree,0.8812,0.1735,0.2005,0.1860,0.5655,0.0889
0,Dummy Classifier,0.9323,0.0000,0.0000,0.0000,0.5000,0.0677


The tuned Random Forest did not improve validation PR-AUC over the baseline Random Forest. Therefore, the baseline Random Forest remains the preferred candidate for final threshold selection.

## Threshold Selection

In [41]:
rf_threshold_results = threshold_metrics(
    "Random Forest", fitted_pipelines["Random Forest"], X_val, y_val,
    thresholds=np.arange(0.08, 0.21, 0.01)
).round(4)

rf_threshold_candidates = rf_threshold_results.sort_values("f1_score", ascending=False).head(8).reset_index(drop=True)
display(rf_threshold_candidates)

,model,threshold,precision,recall,f1_score,predicted_late_orders,actual_late_orders
0,Random Forest,0.16,0.2880,0.3244,0.3051,1472,1307
1,Random Forest,0.14,0.2557,0.3780,0.3050,1932,1307
2,Random Forest,0.13,0.2385,0.4132,0.3024,2264,1307
3,Random Forest,0.12,0.2243,0.4598,0.3016,2679,1307
4,Random Forest,0.18,0.3231,0.2823,0.3013,1142,1307
5,Random Forest,0.15,0.2681,0.3435,0.3011,1675,1307
6,Random Forest,0.17,0.3031,0.2992,0.3011,1290,1307
7,Random Forest,0.19,0.3389,0.2624,0.2958,1012,1307


In [42]:
selected_threshold = rf_threshold_candidates.loc[0, "threshold"]

print(f"Selected threshold: {selected_threshold:.2f}")

Selected threshold: 0.16


The selected threshold is based on the best validation F1-score, balancing late-order detection with alert volume.

## Final Test Evaluation

In [45]:
selected_threshold = 0.16

final_rf_pipeline = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("model", RandomForestClassifier(class_weight="balanced", n_estimators=200, random_state=42, n_jobs=-1))
])

final_rf_pipeline.fit(X_train_val, y_train_val)

test_proba = final_rf_pipeline.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= selected_threshold).astype(int)

final_test_results = pd.DataFrame([{
    "model": "Random Forest",
    "threshold": selected_threshold,
    "accuracy": accuracy_score(y_test, test_pred),
    "precision": precision_score(y_test, test_pred, zero_division=0),
    "recall": recall_score(y_test, test_pred, zero_division=0),
    "f1_score": f1_score(y_test, test_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, test_proba),
    "pr_auc": average_precision_score(y_test, test_proba),
    "predicted_late_orders": test_pred.sum(),
    "actual_late_orders": y_test.sum()
}]).round(4)

display(final_test_results)
display(pd.DataFrame(
    confusion_matrix(y_test, test_pred),
    index=["Actual On Time", "Actual Late"],
    columns=["Predicted On Time", "Predicted Late"]
))

,model,threshold,accuracy,precision,recall,f1_score,roc_auc,pr_auc,predicted_late_orders,actual_late_orders
0,Random Forest,0.16,0.8991,0.2914,0.342,0.3147,0.762,0.2384,1534,1307


,Predicted On Time,Predicted Late
Actual On Time,16900,1087
Actual Late,860,447


The final Random Forest model improves substantially over the baseline late delivery rate. With a selected threshold of 0.16, 29.14% of flagged orders are actually late, compared with a baseline late rate of 6.77%.

The model captures 34.2% of late deliveries while flagging 1,534 orders in the test set. This makes it useful as a risk prioritization tool, although not as a complete late-delivery detection system.

## Save Outputs

In [46]:
outputs_path = project_root / "outputs"
outputs_path.mkdir(exist_ok=True)

final_model_comparison = pd.concat([model_comparison_with_tuned, final_test_results], ignore_index=True)
final_model_comparison.to_csv(outputs_path / "model_comparison_results.csv", index=False)

test_predictions = pd.DataFrame({
    "order_id": test_ids.values,
    "actual_is_late": y_test.values,
    "late_risk_score": test_proba,
    "predicted_is_late": test_pred
})

test_predictions.to_csv(outputs_path / "test_predictions.csv", index=False)
joblib.dump(final_rf_pipeline, outputs_path / "best_model.pkl")

print("Saved outputs:")
print(outputs_path / "model_comparison_results.csv")
print(outputs_path / "test_predictions.csv")
print(outputs_path / "best_model.pkl")

Saved outputs:
c:\Users\johan\Desktop\supplyguard-delivery-risk\outputs\model_comparison_results.csv
c:\Users\johan\Desktop\supplyguard-delivery-risk\outputs\test_predictions.csv
c:\Users\johan\Desktop\supplyguard-delivery-risk\outputs\best_model.pkl


## Final Conclusions

This notebook built a leakage-safe machine learning workflow for late delivery risk prediction.

The final selected model is a Random Forest classifier using a validation-selected threshold of 0.16. On the untouched test set, it achieved 29.14% precision, 34.2% recall, 31.47% F1-score and 23.84% PR-AUC.

The model is not production-ready, but it provides a useful risk-ranking layer for operational prioritization. Its main value is identifying a smaller group of orders with substantially higher late-delivery risk than the dataset baseline.